# llm

> One call shape for each model: `ask(prompt, sp) -> str`.

This module connects the tutor to a model. The tutor needs only one function, `ask(prompt, sp)`, which returns text.
The module also parses JSON replies and gives `Fake`, a model for tests that runs offline.

In [ ]:
#| default_exp llm

In [ ]:
#| hide
import os, tempfile
os.environ.setdefault('LITESEARCH_CACHE', tempfile.mkdtemp(prefix='ls-'))   # and litesearch does not wait on the shared lock of another run
os.environ['ABHYASA_CONFIG'] = tempfile.mkdtemp()   # the settings go to a temporary folder, not to your real settings
os.environ['RAMABANA_CONFIG'] = os.path.join(tempfile.mkdtemp(), 'config.toml')   # and ramabana reads no real config

In [ ]:
#| hide
from fastcore.test import *

In [ ]:
#| hide
import os, tempfile
os.environ['VISHALAKSHI_VAULT'] = os.environ['ABHYASA_VAULT'] = os.path.join(tempfile.mkdtemp(), 'vault.db')   # a test that opens the default vault gets a temporary one

Import the tools that this module uses.

In [ ]:
#| export
import json, os, re
from fastcore.utils import *
from abhyasa.settings import val, dflt, fit, settings, SETTINGS, BORROWS

## The model

Each surface gives its own `ask`.

| surface | `ask` |
|---|---|
| web app | `route_ask`: each call goes through the routing of ramabana, on a backend that stays open |
| notebook | `rishi_ask`: a new `Chat` of the model layer for each call |
| the ramabana terminal agent | `agent.oneshot`, so the routing of the agent applies |
| tests | `Fake` |

The default model is `--model`, else `$ABHYASA_MODEL`, else `DFLT_MODEL` (`claude/sonnet`).

In [ ]:
#| export
DFLT_MODEL = 'claude/sonnet'

def dflt_model(model:str=None) -> str:
    "`model`, else `$ABHYASA_MODEL`, else the setting `models.default`, else `DFLT_MODEL`."
    return model or os.getenv('ABHYASA_MODEL') or val('models.default') or DFLT_MODEL

def rishi_ask(model:str=None):
    "An `ask(prompt, sp='')` on a new `rishi.Chat` (the model layer) for each call, on `dflt_model(model)`."
    from rishi import Chat, resp_text
    model = dflt_model(model)
    def ask(prompt, sp=''):
        chat = Chat(model, sp=sp)
        try: return resp_text(chat(prompt))
        finally: chat.close()
    ask.model = model
    return ask

## JSON replies

`parse_json` finds the JSON in a reply. `ask_json` asks again one time when the reply is not valid JSON.

In [ ]:
#| export
def parse_json(s:str):
    "The first JSON object or array in `s`, also inside a ``` fence. Raise `ValueError` when there is none."
    s = s.strip()
    dec = json.JSONDecoder()
    # the whole reply first, then each fence, then the first object or array in the text: a fence inside a JSON string stays whole
    for t in [s, *(m.strip() for m in re.findall(r'```(?:json)?\s*(.*?)```', s, re.S))]:
        try: return json.loads(t)
        except json.JSONDecodeError: pass
    for i, ch in enumerate(s):
        if ch in '{[':
            try: return dec.raw_decode(s, i)[0]
            except json.JSONDecodeError: continue
    raise ValueError(f'no JSON in the reply: {s[:200]!r}')

def ask_json(ask, prompt:str, sp:str='', tries:int=2):
    "Ask for JSON and parse it. On a bad reply, ask one more time with the parse error."
    p = prompt + '\n\nReply with only one JSON value and no other text.'
    for i in range(tries):
        out = ask(p, sp)
        try: return parse_json(out)
        except ValueError as e:
            if i == tries - 1: raise
            p = f'{prompt}\n\nYour last reply was not valid JSON ({e}). Reply with only one JSON value.'

`parse_json` reads JSON in a fence, JSON inside other text, and arrays. A fence inside a JSON string does not break it. It fails when there is no JSON.

In [ ]:
test_eq(parse_json('```json\n{"a": 1}\n```'), {'a': 1})
test_eq(parse_json('Here it is: {"a": [1, 2]} thanks'), {'a': [1, 2]})
test_eq(parse_json('[{"x": 1}]'), [{'x': 1}])
test_fail(lambda: parse_json('nothing'), contains='no JSON')
test_eq(parse_json('{"answer": "Use ```python\\nx = 1\\n``` here."}'), {'answer': 'Use ```python\nx = 1\n``` here.'})
test_eq(parse_json('Sure. {"a": 1} and then {"b": 2}'), {'a': 1})

## A model for tests

`Fake` replies from a dict of rules. The first rule with a key in the prompt gives the reply.
It sends a dict or a list reply as JSON. It keeps each call in `calls`.

In [ ]:
#| export
class Fake:
    "A stand-in model for tests: `Fake({'grade': {...}, 'question': {...}})`."
    def __init__(self, rules:dict=None, dflt='{}'): self.rules, self.dflt, self.calls = rules or {}, dflt, []
    def __call__(self, prompt, sp=''):
        self.calls.append((prompt, sp))
        r = next((v for k, v in self.rules.items() if k in prompt or k in sp), self.dflt)
        if callable(r): r = r(prompt, sp)
        return r if isinstance(r, str) else json.dumps(r)

A matching rule gives its reply, and `Fake` records the call.

In [ ]:
f = Fake({'GRADE': dict(correct=True)})
test_eq(ask_json(f, 'GRADE this'), dict(correct=True))
test_eq(len(f.calls), 1)

After a reply that is not JSON, `ask_json` asks again and gets the JSON.

In [ ]:
bad = Fake(dict(X=lambda p, sp: 'oops' if 'not valid' not in p else '{"ok": 1}'))
test_eq(ask_json(bad, 'X'), {'ok': 1})

## A model for each task

The tutor does five tasks, and each task can use its own model. A quick model can grade and give hints. A strong model can
write questions and lessons. The Settings page saves the choice. `Router` reads it on each call, so a change has an effect on the next call.

| task | what the model does |
|---|---|
| `question` | writes a question and its hidden model answer |
| `grade` | reads the conversation and grades it |
| `hint` | gives the next hint |
| `lesson` | teaches the smallest step after a miss |
| `extract` | finds the concepts in a text |

Each task also has a reply cap: the most tokens that the model can write in one reply. `TOKENS` gives the default of each task.
A question has options and a rubric, and an extract can name 15 concepts, so these two get the most. A hint is short. The Settings
page can change each cap. Only the cap is a setting, because ramabana's `oneshot` uses no other per-call value: it turns thinking
off, and the time limit of a model is in ramabana's `config.toml`. The model layer gives the cap to the API models (OpenAI, Codex,
Gemini, Copilot) and to the models on this Mac. Claude Code does not use it for the reply, but ramabana uses it to fit the prompt
in the window of the model.

In [ ]:
#| export
TASKS = ('question', 'grade', 'hint', 'lesson', 'extract')
TOKENS = {t: dflt(f'tokens.{t}') for t in TASKS}   # the reply cap of each task, when the settings give none
TOKENS_MAX = SETTINGS['tokens.question'].hi

`model_ask` gives an `ask` for any model name. Install the `models` extra (`pip install abhyasa[models]`) to use the full list of models.
With the extra, the name can be any model in that list: Claude Code, Codex, OpenAI, Copilot, Gemini, or a model on this machine (LiteRT, mlx,
llama.cpp, Ollama). The call goes through the routing of ramabana, as `route_ask` below shows. Without the extra, the name goes to the model layer as it is.

In [ ]:
#| export
import threading, time
_asks, _asks_lock = {}, threading.Lock()

def model_ask(name:str):
    "An `ask(prompt, sp='')` for model `name`, made once and kept."
    with _asks_lock:
        if name not in _asks: _asks[name] = route_ask(name)
        return _asks[name]

## Names for the model list

The Settings page shows a list of models. Each item has a name that people know, such as "Claude Sonnet" or "GPT-5.5".
The model id, such as `claude/sonnet`, is the value that the settings keep. `model_label` makes the name from the id.
`PROVIDERS` gives the name of each group in the list, and `provider_label` reads it. ramabana's `is_local` tells if a model runs on this Mac.
For Ollama, it looks at the daemon host (the `host` in config.toml, else `$OLLAMA_HOST`), and it does not connect to it.
Thus a model on an Ollama daemon on another computer has no "a local model" name, and its group is "Ollama on another computer". Without the `models` extra, no local model is in the list, so no name says "a local model".

In [ ]:
#| export
PROVIDERS = dict(claude='Claude', anthropic='Anthropic API', openai='OpenAI', codex='Codex', gemini='Gemini', copilot='GitHub Copilot',
                 azure='Azure', remote='Other hosted models', litert='On this Mac (LiteRT)', mlx='On this Mac (MLX)',
                 llama='On this Mac (llama.cpp)', ollama='On this Mac (Ollama)')
_DROP = re.compile(r'^(?:mlx|gguf|it|litert|lm|optiq|instruct|\d+bit)$', re.I)

def _words(m:str) -> str:
    "`claude-opus-5-5` gives `Claude Opus 5.5`, `gpt-6-astra` gives `GPT-6 Astra`, `Qwen3-4B` gives `Qwen 3 4B`."
    m = re.sub(r'-20\d{6}$', '', m)
    out = []
    for t in (t for t in re.split(r'[-_:\s]+', m) if t and not _DROP.match(t)):
        if t.isdigit() and out and re.fullmatch(r'[\d.]+', out[-1]) and '.' not in out[-1]: out[-1] += '.' + t
        elif re.fullmatch(r'[a-z]?[\d.]+[bmk]', t, re.I): out.append(t.upper())
        elif t.lower() == 'gpt': out.append('GPT')
        elif out and out[-1].startswith('GPT') and re.fullmatch(r'[\d.]+', t) and out[-1] == 'GPT': out[-1] += f'-{t}'
        else: out.append(re.sub(r'^([A-Za-z]+)(\d)', r'\1 \2', t[0].upper() + t[1:]))
    return ' '.join(out)

def _on_mac(value:str, provider:str='') -> bool:
    "Whether the model `value` of `provider` runs on this Mac: ramabana's `is_local`. False with no ramabana."
    try: from ramabana.core import is_local
    except ImportError: return False
    return is_local(value) or bool(provider) and is_local(f'{provider}/{value}')

def model_label(value:str, provider:str='', repo:str='') -> str:
    "A name that people know for the model id `value`; `repo` is the download of a local model, if known."
    pre, _, m = value.rpartition('/')
    provider = provider or pre
    if provider in ('remote', 'azure'): return value   # a name that you gave in ramabana's config.toml
    name = _words(repo.rpartition('/')[2] if repo else m) or value
    if provider in ('claude', 'anthropic') and not name.startswith('Claude'): name = f'Claude {name}'
    return f'a local model: {name}' if _on_mac(value, provider) else name

def provider_label(provider:str, values=()) -> str:
    "The name of the `provider` group in the model list, for the model ids `values`. An Ollama daemon on another computer is not on this Mac."
    if provider == 'ollama' and values:
        here = {_on_mac(v, provider) for v in values}
        if here == {False}: return 'Ollama on another computer'
        if here == {True, False}: return 'Ollama'
    return PROVIDERS.get(provider, provider.capitalize() or 'Other')

Each kind of id gets a plain name. A Claude Code tier gets the word Claude. A local model says so.

In [ ]:
test_eq(model_label('claude/sonnet'), 'Claude Sonnet')
test_eq(model_label('claude/claude-opus-5-5'), 'Claude Opus 5.5')
test_eq(model_label('anthropic/claude-haiku-4-5-20251001'), 'Claude Haiku 4.5')
test_eq(model_label('codex/gpt-6-astra'), 'GPT-6 Astra')
test_eq(model_label('openai/gpt-5.4-mini'), 'GPT-5.4 Mini')
test_eq(model_label('gemini/gemini-3.5-flash'), 'Gemini 3.5 Flash')
test_eq(model_label('qwen-4b', 'mlx', 'mlx-community/Qwen3.5-4B-MLX-4bit'), 'a local model: Qwen 3.5 4B')
test_eq(model_label('gemma-e4b', 'litert', 'litert-community/gemma-4-E4B-it-litert-lm'), 'a local model: Gemma 4 E4B')
test_eq(model_label('my-model', 'azure'), 'my-model')

An Ollama model is local only when its daemon is on this Mac. With `$OLLAMA_HOST` on localhost, the model and its group are on this Mac:

In [ ]:
from fastcore.xtras import modified_env
with modified_env(OLLAMA_HOST='localhost:11434'):
    test_eq(model_label('ollama/qwen3:8b'), 'a local model: Qwen 3 8B')
    test_eq(provider_label('ollama', ['ollama/qwen3:8b']), 'On this Mac (Ollama)')

With `$OLLAMA_HOST` on another computer, the model is not called local, and its group has a hosted name. The check reads only the host name: it does not connect:

In [ ]:
with modified_env(OLLAMA_HOST='http://gpu-box.lan:11434'):
    test_eq(model_label('ollama/qwen3:8b'), 'Qwen 3 8B')
    test_eq(provider_label('ollama', ['ollama/qwen3:8b']), 'Ollama on another computer')

The other groups keep their names from `PROVIDERS`. A provider that is not in it gets its own name:

In [ ]:
test_eq(provider_label('mlx', ['qwen-4b']), 'On this Mac (MLX)')
test_eq(provider_label('claude', ['claude/sonnet']), 'Claude')
test_eq(provider_label('groq'), 'Groq')
test_eq(provider_label(''), 'Other')

`catalog` lists the models that can run here, for the Settings page. With the `models` extra, the list is the full list of models. Each row gets a `name` from `model_label`. The list stays in memory for five minutes, because the look-up asks each provider. Without the extra, the list has only the models in `FALLBACK`.

In [ ]:
#| export
FALLBACK = ['claude/sonnet', 'claude/haiku', 'claude/opus']
_cat = {}

def _repos():
    "The download of each local model in the full model list, for the names."
    try:
        from ramabana.core import LOCAL as lr, MLX, LLAMA
        return {**lr, **MLX, **LLAMA}
    except Exception: return {}

def catalog(ttl:float=300.) -> list:
    "The models that can run here, as dicts with `value`, `name`, `label`, `provider` and `source`."
    if _cat.get('at', 0) > time.time() - ttl: return _cat['rows']
    try:
        from ramabana.core import available_models
        rows = available_models()
    except Exception: rows = [dict(value=m, label=m, provider='claude', source='rishi') for m in FALLBACK]
    repos = _repos()
    rows = [dict(r, name=model_label(r['value'], r.get('provider', ''), repos.get(r['value'], ''))) for r in rows]
    _cat.update(rows=rows, at=time.time())
    return rows

Without the `models` extra, the list has the three Claude tiers, each with a name.

In [ ]:
_cat.clear()
try: import ramabana; has_rama = True
except ImportError: has_rama = False
if not has_rama: test_eq([r['name'] for r in catalog()], ['Claude Sonnet', 'Claude Haiku', 'Claude Opus'])
assert all(r['name'] for r in catalog())

`Router` is the `ask` the app gives the tutor. Called as it is, it uses the default model. `for_(task)` gives the `ask` for one
task: the model the settings name for it, else the default. `for_(task, name, job)` gives the `ask` for `task` on the model `name`, on the
ramabana job `job`. The Test button of the Settings page uses the job `test:<task>`, so a test does not move the job of the task. When the default routes by task, as `RouteAsk` below does, `for_` lets it make the `ask`.
`cap(task)` gives the reply cap of the task, read on each call. `warm` starts the backend of each task in the background.

In [ ]:
#| export
ALL_TASKS = TASKS + tuple(BORROWS)   # the tasks after `extract` borrow a model and a reply cap: `settings.BORROWS`
WARM = dict(guide='hint')   # the jobs that are not tasks and that start with the app: job -> the task whose model it uses

class Router:
    "The tutor's model: a default `ask`, and one model for each task that the settings name."
    def __init__(self, dflt, models=None, tokens=None):
        self.dflt = dflt
        self.models = models or (lambda: settings().get('models', {}))
        self.tokens = tokens or (lambda: settings().get('tokens', {}))
    @property
    def model(self) -> str: return getattr(self.dflt, 'model', '')
    def __call__(self, prompt, sp=''): return self.dflt(prompt, sp)
    def own(self, task:str) -> str:
        "The model saved for `task` itself, or ''."
        n = (self.models() or {}).get(task)
        return n if isinstance(n, str) else ''
    def name(self, task:str) -> str:
        "The model for `task`: its own, else the model of the task it borrows from, else '' for the default."
        return self.own(task) or (self.own(BORROWS[task]) if task in BORROWS else '')
    def cap(self, task:str) -> int:
        "The reply cap of `task` (or of the task it borrows from): the saved value made to fit its range, else `TOKENS`."
        t = BORROWS.get(task, task)
        return fit(SETTINGS[f'tokens.{t}'], (self.tokens() or {}).get(t))
    def for_(self, task:str, name:str=None, job:str=None):
        "The `ask` for `task` on `name`, else on the model of the settings, else on the default. A routed call goes to the job `job`, else `task`. A task with no model of its own uses the job of the task it borrows from."
        n = self.name(task) if name is None else name
        job = job or (task if task not in BORROWS or self.own(task) else BORROWS[task])
        if hasattr(self.dflt, 'via'): return self.dflt.via(job, n, self.cap(task))
        return model_ask(n) if n else self.dflt
    def warm(self, jobs=None):
        "Start the backend of each job in `jobs` (default: each task, and each job of `WARM`) in a background thread, with no model call. Returns the thread, or None."
        if not hasattr(self.dflt, 'warm'): return None
        names = {j: self.name(WARM.get(j, j)) for j in (TASKS + tuple(WARM) if jobs is None else jobs)}
        t = threading.Thread(target=lambda: self.dflt.warm(names), daemon=True)
        t.start()
        return t

def for_task(ask, task:str):
    "The `ask` for `task` when `ask` routes by task, else `ask` itself."
    return ask.for_(task) if hasattr(ask, 'for_') else ask

A router with no settings uses the default for every task. A task that names a model gets that model.

In [ ]:
dflt = Fake(dflt='default')
r = Router(dflt, models=lambda: {})
test_eq(r('x'), 'default')
test_is(r.for_('grade'), dflt)
named = Fake(dflt='named'); named.model = 'fast'
_asks['fast'] = named
r = Router(dflt, models=lambda: dict(grade='fast'))
test_eq(r.for_('grade')('x'), 'named')
test_eq(r.for_('question')('x'), 'default')
test_is(for_task(dflt, 'grade'), dflt)

## Every call through ramabana

ramabana is the agent layer on top of the model layer. Its `Routing` sends each job to a model. When a cloud model fails or
gives no answer, `Routing` puts it aside for five minutes and sends the job to the next route: other cloud models, then a small model
on this Mac. Its backends stay open, so a call does not pay the start time of a new `Chat`.

`ramabana_agent` makes one ramabana `Agent` for the process, on the first call. The `Agent` gets only what `Agent.oneshot` needs:

- `NullHost`: no folders, and each file operation is refused.
- `readonly=True`, no sub-agents, no extensions, no skills, no lessons: no tool can write, run code or go on the network.
- Approvals that refuse when nobody answers, and no saved rules.
- `cfg=None`: no history, plan or run log, so the `Agent` writes no files.
- A short briefing, so that it does not read skills or project files.

The turn of the `Agent` is the default model, so a task whose model fails goes to the default model first.

In [ ]:
#| export
import warnings, importlib.util
MAX_TOKENS = 4096   # the reply cap of a call with no task; ramabana's own cap of 1024 can cut a long reply
_ra, _ra_lock, _locks = {}, threading.RLock(), {}

def ramabana_agent(turn:str=None):
    "The one ramabana `Agent` of this process, made on first use on the turn `dflt_model(turn)`: no folders, no tool that acts, no files."
    with _ra_lock:
        if 'agent' not in _ra:
            from ramabana.agent import Agent, Approvals
            from ramabana.core import Routing
            from ramabana.tools import NullHost
            _ra['agent'] = Agent(NullHost(), routing=Routing(turn=dflt_model(turn)), cfg=None, sp='You are the model of abhyasa, a tutor.',
                                 approvals=Approvals(mode='ask', no_listener='refuse', timeout=0), readonly=True, subagents=False,
                                 extensions=False, global_skills=False, learned=False, compact=False, poll_every=0, auto_turns='off')
        return _ra['agent']

Each task is a ramabana job with the same name. `_point` sets the model of the job before each call, only when the settings changed it.
A model that ramabana put aside stays aside until its time is up.

In [ ]:
#| export
def _point(a, job:str, name:str):
    "Point the ramabana job `job` at the model `name`, when it is not there yet."
    with _ra_lock:
        if a.routing.policy.get(job) == name: return
        try: a.routing.set(name, job=job)
        except Exception as e: raise RuntimeError(f'The {job} task: ramabana does not know the model {name!r} ({e}).') from e

def _lock(b):
    "The abhyasa lock of the backend `b`: one call at a time on each backend."
    with _ra_lock: return _locks.setdefault((b.spec.backend, b.spec.model_id), threading.Lock())

def _why(a, job:str, since:float) -> str:
    "The notes of ramabana on why `job` got no answer: the models it put aside since `since`, else the note of the route."
    notes = [f'{n} {w}' for n, (w, t) in a.routing.down.items() if t >= since] or [a.routing.notes.get(job, '')]
    return '; '.join(n for n in notes if n) or 'no note'

`ramabana_oneshot` asks one question on the job of a task. ramabana's `Backend.oneshot` does not wait when another call holds the
backend: it gives ''. So the call holds the abhyasa lock of each backend that it uses. `Agent.oneshot` calls `fits(b)` just before
it asks the backend `b`, on the first route and on each fallback. `fits` lets go of the lock of the route before, then waits for the
lock of `b`. A call holds one lock at a time, so two calls cannot wait for each other. When no route gives an answer, the call raises.

A failure can take some time: ramabana tries each route in turn, then the small model on this Mac. It puts a failed cloud model aside for five minutes.

A route can also raise ramabana's `OneshotTimeout` when a call goes past its deadline. `timeout_error` changes it into the same
`RuntimeError` as a call with no answer. The error names the model and the seconds, so each caller catches one kind of error.

`prepare` lets the caller change the text for each route, as ramabana's `Agent.oneshot` does. The guide uses it for its guard:
the guard then knows if the model of the route is on this Mac. `HOSTED` is the model that a guard takes when it does not know
the route.

In [ ]:
#| export
def timeout_error(job:str, e, name:str='') -> RuntimeError:
    "The `RuntimeError` for the ramabana `OneshotTimeout` `e` on the job `job`: it names the model (`name`, else the route of `e`) and the seconds."
    return RuntimeError(f'The {job} task got no answer from {name or e.route} through ramabana: {e.route} did not answer in {e.elapsed:.0f} seconds.')

HOSTED = AttrDict(name='hosted', local=False)   # the model that a guard takes when it does not know the route: one off this Mac

def _one_lock():
    "A `fits(b)` for ramabana's one-shot calls, and `let_go()`, which lets go of the lock that it holds."
    held = []
    def let_go():
        if held: held.pop().release()
    def fits(b):
        "Let go of the lock of the route before, then wait for the lock of `b`: one lock at a time, so no deadlock."
        let_go(); lk = _lock(b); lk.acquire(); held.append(lk)
        return True
    return fits, let_go

def ramabana_oneshot(job:str, name:str, prompt:str, sp:str='', max_tokens:int=MAX_TOKENS, prepare=None) -> str:
    "Ask `prompt` on the ramabana job `job`, pointed at the model `name`. Wait for a busy backend. Raise when no route answers. `prepare(prompt, sp, backend)` gives the text for each route, so a guard knows the model that gets it."
    from ramabana.runtime import OneshotTimeout
    a, since = ramabana_agent(name), time.time()
    _point(a, job, name)
    fits, let_go = _one_lock()
    try: out = a.oneshot(prompt, sp, job=job, max_tokens=max_tokens, fits=fits, **({'prepare': prepare} if prepare else {}))
    except OneshotTimeout as e: raise timeout_error(job, e, name) from e
    finally: let_go()
    if out: return out
    raise RuntimeError(f'The {job} task got no answer from {name} through ramabana: {_why(a, job, since)}')

`ramabana_stream` is the twin of `ramabana_oneshot` that gives the reply as the model sends it, through ramabana's
`Agent.oneshot_stream`. It holds the same lock of each backend, and it raises the same errors. The guard (`prepare`) runs
before the stream starts. A route that cannot stream (ramabana's `StreamUnsupported`, before it sends anything) gives the whole
reply of `ramabana_oneshot` as one chunk, so the caller does not need a second path.

In [ ]:
#| export
def ramabana_stream(job:str, name:str, prompt:str, sp:str='', max_tokens:int=MAX_TOKENS, prepare=None):
    "The reply of `ramabana_oneshot`, one chunk at a time as the model sends it. A route that cannot stream gives the whole reply as one chunk. Raise when no route answers."
    from ramabana.runtime import OneshotTimeout, StreamUnsupported
    a, since, said = ramabana_agent(name), time.time(), False
    _point(a, job, name)
    fits, let_go = _one_lock()
    try:
        try:
            for d in a.oneshot_stream(prompt, sp, job=job, max_tokens=max_tokens, fits=fits, **({'prepare': prepare} if prepare else {})):
                said = said or bool(d)
                yield d
        except StreamUnsupported:
            let_go()
            yield ramabana_oneshot(job, name, prompt, sp, max_tokens, prepare)
            return
        except OneshotTimeout as e: raise timeout_error(job, e, name) from e
    finally: let_go()
    if not said: raise RuntimeError(f'The {job} task got no answer from {name} through ramabana: {_why(a, job, since)}')

`RouteAsk` is the default `ask` of the web app. `via(task, name, max_tokens)` gives the `ask` of one task, and `Router.for_` uses it. A call with
no task goes to the job `ask:<model>`. `warm` starts the backend of each task, with no model call. `route_ask` gives a `RouteAsk`
when ramabana is installed, else `rishi_ask` with a warning.

In [ ]:
#| export
class RouteAsk:
    "The default `ask`: each call goes through the routing of ramabana, on the job of its task."
    def __init__(self, model:str=None): self._model = model
    @property
    def model(self) -> str:
        "The default model now: `model`, else `$ABHYASA_MODEL`, else the setting `models.default`, else `DFLT_MODEL`."
        return dflt_model(self._model)
    def via(self, task:str, name:str='', max_tokens:int=None, prepare=None):
        "The `ask` for `task` on the model `name`, else on the default model, with the reply cap `max_tokens`, else the default of `task`. `prepare` goes to `ramabana_oneshot`. `ask.stream(prompt, sp)` gives the reply in chunks (`ramabana_stream`)."
        name, cap = name or self.model, max_tokens or TOKENS.get(task, MAX_TOKENS)
        def ask(prompt, sp=''): return ramabana_oneshot(task, name, prompt, sp, cap, prepare)
        ask.stream = lambda prompt, sp='': ramabana_stream(task, name, prompt, sp, cap, prepare)
        ask.model = name
        return ask
    def __call__(self, prompt, sp=''): return self.via(f'ask:{self.model}')(prompt, sp)
    def warm(self, names:dict):
        "Make the `Agent` and start the backend of each task in `names` (task -> model, '' for the default). No model call."
        a = ramabana_agent(self.model)
        for task, name in names.items():
            try:
                _point(a, task, name or self.model)
                a.warm_backend(task)   # starts the backend under the lock of the `Agent`, so a call at the same time waits for it
            except Exception as e: warnings.warn(f'abhyasa could not start the model of the {task} task: {e}')

def _has_ramabana() -> bool:
    try: return importlib.util.find_spec('ramabana') is not None
    except (ImportError, ValueError): return False

def route_ask(model:str=None):
    "The default `ask`: a `RouteAsk` when ramabana is installed, else `rishi_ask`."
    if _has_ramabana(): return RouteAsk(model)
    warnings.warn('abhyasa: ramabana (the models extra) is not installed. Each call uses a new rishi Chat, which is slower.')
    return rishi_ask(model)

### The worker agent

`ramabana_worker` makes a second ramabana `Agent`, for the gather agent of research. It is never the one `Agent` of `ramabana_agent`, so a long gather never holds a one-shot call. It runs on the default routing (`Routing(turn=dflt_model())`). Its host is the shalya `host` (default: a `NullHost`, with no folders). The caller names its tools in `tools`, then adds them with `add_tool`. ramabana's `only_tools` offers the model only those tools, and refuses a tool with another name. ramabana's memory and plan tools are held back. It has no approvals listener, so a tool that writes is refused. The caps stop a turn after `max_calls` tool calls, `max_steps` model steps, `max_tokens` tokens or `max_cost` dollars. A model such as Claude Code on a plan reports no cost. Then ramabana estimates the dollars from the tokens at the price of the model (`ramabana.agent.token_rate`), and keeps the estimate in `estimated`, so the dollar cap still holds.

In [ ]:
#| export
WORKER_SP = 'You are the source gatherer of abhyasa, a tutor.'
WORKER_CALLS, WORKER_STEPS, WORKER_DOLLARS = 300, 80, 5.   # the caps of one turn: tool calls, model steps, dollars
WORKER_TOKENS = 2_000_000   # the token cap of one turn, a backstop: the dollar cap, from the price of the model, stops a turn first

def ramabana_worker(tools:list=(), host=None, sp:str=WORKER_SP, turn:str=None, max_calls:int=WORKER_CALLS, max_steps:int=WORKER_STEPS,
                    max_cost:float=WORKER_DOLLARS, max_tokens:int=WORKER_TOKENS):
    "A new ramabana `Agent` on the turn `dflt_model(turn)`, with the shalya `host` (default: no folders). The model sees only the tools named in `tools`, when the caller adds them. Never the agent of `ramabana_agent`. A turn stops at `max_tokens` tokens, and at `max_cost` dollars (estimated from the tokens when the model reports no cost)."
    from ramabana.agent import Agent, Approvals
    from ramabana.core import Routing
    from ramabana.tools import NullHost
    return Agent(host or NullHost(), routing=Routing(turn=dflt_model(turn)), cfg=None, sp=sp, only_tools=list(tools),
                 approvals=Approvals(mode='ask', no_listener='refuse', timeout=0), readonly=True, subagents=False, extensions=False,
                 global_skills=False, learned=False, compact=False, poll_every=0, auto_turns='off',
                 max_tool_calls=max_calls, max_steps=max_steps, max_cost=max_cost, max_tokens=max_tokens)

The worker offers only the tool that the test names and adds. It is a new `Agent` each time, and never the shared one. The caps reach the `Agent`.

In [ ]:
if _has_ramabana():
    def look(q:str, # what to look for
             ) -> str:
        "Look for `q`."
        return q
    w = ramabana_worker(['look'], max_calls=50, max_tokens=1000)
    w.add_tool(look)
    test_eq([t.__name__ for t in w.tools], ['look'])
    assert w is not ramabana_worker() and w is not ramabana_agent()
    test_eq((w.readonly, w.subagents, w.tool_budget, w.max_cost, w.max_tokens), (True, False, 50, WORKER_DOLLARS, 1000))

The worker refuses a tool that `tools` does not name. A worker with no names offers no tool, so ramabana's own tools stay out.

In [ ]:
if _has_ramabana():
    def write(path:str, # the file
              ) -> str:
        "Write `path`."
        return path
    test_fail(lambda: w.add_tool(write), contains='not in only_tools')
    test_eq(ramabana_worker().tools, [])

When the model reports no cost, ramabana estimates the dollars from the tokens with its own price table (`est_rate` is None), and
keeps them in `estimated`. A new worker has estimated nothing, and has the default token cap.

In [ ]:
if _has_ramabana():
    w = ramabana_worker()
    test_eq((w.est_rate, w.estimated, w.max_tokens), (None, 0., WORKER_TOKENS))

### Tests of the route

These tests use a ramabana `Agent` with fake backends from `ramabana.testing`, so no model runs. `using(a)` puts that `Agent` in
place of the real one for one test. CI does not install the `models` extra, so these tests run only when ramabana is installed.

In [ ]:
from contextlib import contextmanager
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
if has_rama:
    from ramabana.core import resolve
    from ramabana.testing import fake_agent, FakeBackend

@contextmanager
def using(a):
    "Use the `Agent` `a` in place of the real one, for one test."
    _ra['agent'] = a
    try: yield a
    finally: _ra.pop('agent', None)

Each task goes to the ramabana job with the same name. The task that the settings name gets its model, the other tasks get the
default. Each call asks for the reply cap of its task, `TOKENS[task]`, so a long question with its rubric is not cut.

In [ ]:
if has_rama:
    a, be = fake_agent()
    jobs, one = [], a.oneshot
    a.oneshot = lambda p, sp='', job='oneshot', max_tokens=None, fits=None: jobs.append((job, max_tokens)) or one(p, sp, job, max_tokens, fits)
    with using(a):
        r = Router(RouteAsk('claude/sonnet'), models=lambda: dict(grade='claude/haiku'), tokens=lambda: {})
        test_eq(r.for_('grade')('hi'), 'ONESHOT:hi')
        test_eq(r.for_('question').model, 'claude/sonnet')
        r.for_('question')('hi')
    test_eq(jobs, [('grade', TOKENS['grade']), ('question', TOKENS['question'])])
    test_eq((a.routing.policy['grade'], a.routing.policy['question']), ('claude/haiku', 'claude/sonnet'))

A change in the settings has an effect on the next call, with no restart. An empty choice goes back to the default.

In [ ]:
if has_rama:
    a, be = fake_agent()
    ms = dict(grade='claude/haiku')
    with using(a):
        r = Router(RouteAsk('claude/sonnet'), models=lambda: ms)
        r.for_('grade')('hi'); test_eq(a.routing.policy['grade'], 'claude/haiku')
        ms['grade'] = 'claude/opus'; r.for_('grade')('hi'); test_eq(a.routing.policy['grade'], 'claude/opus')
        ms['grade'] = ''; r.for_('grade')('hi'); test_eq(a.routing.policy['grade'], 'claude/sonnet')

`prepare` gets each route. Here it writes the name of the model before the prompt, so the reply shows what the route got:

In [ ]:
if has_rama:
    a, be = fake_agent()
    with using(a): test_eq(RouteAsk('claude/sonnet').via('hint', prepare=lambda p, sp, b: (f'{b.spec.name} {p}', sp))('hi'), 'ONESHOT:fake hi')

`Slow` is a fake backend that takes a moment to answer. It sets `inside` when a call starts.

In [ ]:
if has_rama:
    class Slow(FakeBackend):
        "A fake backend that answers after 0.2 s and sets `inside` when a call starts."
        def __init__(self, *a, **kw): super().__init__(*a, **kw); self.inside = threading.Event()
        def _oneshot(self, prompt, sp, max_tokens): self.inside.set(); time.sleep(0.2); return f'ONESHOT:{prompt}'

This is why the route has a lock. ramabana's `Agent.oneshot` does not wait for a busy backend: a second call at the same time gets ''.

In [ ]:
if has_rama:
    a, _ = fake_agent(); slow = Slow(); a._be_or_none = lambda job='turn': slow
    with ThreadPoolExecutor(2) as ex:
        f1 = ex.submit(a.oneshot, 'one'); slow.inside.wait(); f2 = ex.submit(a.oneshot, 'two')
    test_eq((f1.result(), f2.result()), ('ONESHOT:one', ''))

Through the route, the second call waits for the first, and both calls get an answer.

In [ ]:
if has_rama:
    slow.inside.clear()
    with using(a), ThreadPoolExecutor(2) as ex:
        ask = RouteAsk('claude/sonnet').via('question')
        f1 = ex.submit(ask, 'one'); slow.inside.wait(); f2 = ex.submit(ask, 'two')
        test_eq((f1.result(), f2.result()), ('ONESHOT:one', 'ONESHOT:two'))

When no route gives an answer, the call raises. The error names the task, the model and the note of ramabana, so `ask_json` never gets ''.

In [ ]:
if has_rama:
    class Empty(FakeBackend):
        "A fake cloud backend that answers nothing."
        def _oneshot(self, prompt, sp, max_tokens): return ''
    a, _ = fake_agent(); empty = Empty(resolve('claude/haiku')); a._be_or_none = lambda job='turn': empty
    with using(a):
        ask = Router(RouteAsk('claude/sonnet'), models=lambda: dict(grade='claude/haiku')).for_('grade')
        test_fail(lambda: ask('hi'), contains='grade task')
        try: ask('hi')
        except RuntimeError as e: assert 'claude/haiku' in str(e) and 'answered nothing' in str(e), e

A route that goes past its deadline raises ramabana's `OneshotTimeout`. The call raises the same `RuntimeError`, with the model and the seconds, and keeps the first error as its cause.

In [ ]:
if has_rama:
    from ramabana.runtime import OneshotTimeout
    a, _ = fake_agent()
    def late(*a, **kw): raise OneshotTimeout('claude/haiku', 31.6, True)
    a.oneshot = late
    with using(a):
        try: RouteAsk('claude/haiku').via('grade')('hi'); assert False, 'no error'
        except RuntimeError as e:
            test_eq(str(e), 'The grade task got no answer from claude/haiku through ramabana: claude/haiku did not answer in 32 seconds.')
            assert isinstance(e.__cause__, OneshotTimeout)

### Tests of the stream

`Chunks` is a fake backend that can stream: it sends its reply three letters at a time. `ask.stream` gives the chunks in order,
and their join is the reply. The guard of `prepare` sees the route before the stream starts.

In [ ]:
if has_rama:
    class Chunks(FakeBackend):
        "A fake backend that streams `ONESHOT:<prompt>` in chunks of 3 letters."
        supports_stream = True
        def _oneshot_stream(self, prompt, sp, max_tokens): return (f'ONESHOT:{prompt}'[i:i+3] for i in range(0, len(f'ONESHOT:{prompt}'), 3))
    a, _ = fake_agent(); ch = Chunks(resolve('claude/haiku')); a._be_or_none = lambda job='turn': ch
    with using(a): got = list(RouteAsk('claude/haiku').via('hint', prepare=lambda p, sp, b: (f'{b.spec.name} {p}', sp)).stream('hi'))
    test_eq(''.join(got), 'ONESHOT:claude/haiku hi')
    assert len(got) > 1

A backend that cannot stream (the default of `FakeBackend`) gives the whole reply of the blocking call as one chunk:

In [ ]:
if has_rama:
    a, be = fake_agent()
    with using(a): test_eq(list(RouteAsk('claude/sonnet').via('hint').stream('hi')), ['ONESHOT:hi'])

The stream lets go of the lock of its backend at the end, so the next call on that backend does not wait:

In [ ]:
if has_rama:
    a, _ = fake_agent(); a._be_or_none = lambda job='turn': ch
    with using(a):
        list(RouteAsk('claude/haiku').via('hint').stream('one'))
        assert _lock(ch).acquire(timeout=1); _lock(ch).release()
        it = RouteAsk('claude/haiku').via('hint').stream('two'); next(it); it.close()   # a stream that stops early lets go too
        assert _lock(ch).acquire(timeout=1); _lock(ch).release()

A stream with no text raises the same error as a blocking call, and a timeout raises the same `RuntimeError`:

In [ ]:
if has_rama:
    a, _ = fake_agent(); empty = Empty(resolve('claude/haiku')); empty.supports_stream = True
    empty._oneshot_stream = lambda prompt, sp, max_tokens: (c for c in ())
    a._be_or_none = lambda job='turn': empty
    with using(a): test_fail(lambda: list(RouteAsk('claude/haiku').via('grade').stream('hi')), contains='grade task')
    def late(*a, **kw): raise OneshotTimeout('claude/haiku', 31.6, True); yield
    a.oneshot_stream = late
    with using(a): test_fail(lambda: list(RouteAsk('claude/haiku').via('grade').stream('hi')), contains='did not answer in 32 seconds')

The next tests need ramabana's real `Routing`, so that a failed model goes to its next route. `routed_agent` gives a fake `Agent` with
the real `Routing` on two fake models: `fake-main`, the default, and `fake-quick`. They are hosted models with a fake key, so
`Routing` takes them on any machine. `make(spec)` gives the fake backend of each model, and `bes` keeps them by name.

In [ ]:
if has_rama:
    from ramabana.core import register_model, Routing
    os.environ['ABHYASA_FAKE_KEY'] = 'not a real key'
    for n in ('fake-main', 'fake-quick'): register_model(n, f'fakevendor/{n}', runtime='remote', api_key_env='ABHYASA_FAKE_KEY')

    def routed_agent(make):
        "A fake `Agent` on the real `Routing`, with the turn on `fake-main`. Returns the `Agent` and the backends by model name."
        a, _ = fake_agent(routing=Routing(turn='fake-main'))
        del a._be_or_none   # the real one, which asks `Routing`
        bes = {}
        def be(job='turn'):
            s = a.routing.spec(job)
            if s.name not in bes: bes[s.name] = make(s)
            return bes[s.name]
        a._be = be
        return a, bes

A fallback holds the lock of the backend that it goes to. Here `fake-quick` answers nothing, so the `grade` call goes to `fake-main`.
A `question` call on `fake-main` starts while that fallback runs. It waits for the lock, and both calls get an answer.

In [ ]:
if has_rama:
    a, bes = routed_agent(lambda s: Slow(s) if s.name != 'fake-quick' else Empty(s))
    with using(a), ThreadPoolExecutor(2) as ex:
        g = ex.submit(Router(RouteAsk('fake-main'), models=lambda: dict(grade='fake-quick'), tokens=lambda: {}).for_('grade'), 'g')
        while 'fake-main' not in bes: time.sleep(0.01)
        bes['fake-main'].inside.wait()
        q = ex.submit(RouteAsk('fake-main').via('question'), 'q')
        test_eq((g.result(), q.result()), ('ONESHOT:g', 'ONESHOT:q'))

When no route answers, each route is tried one time. The last route is the small model on this Mac, `gemma-e4b`.

In [ ]:
if has_rama:
    calls = Counter()
    class Counted(Empty):
        "A fake backend that answers nothing and counts its calls."
        def _oneshot(self, prompt, sp, max_tokens): calls[self.spec.name] += 1; return ''
    a, bes = routed_agent(Counted)
    with using(a): test_fail(lambda: RouteAsk('fake-main').via('grade', 'fake-quick')('hi'), contains='grade task')
    test_eq(calls['gemma-e4b'], 1)
    test_eq(set(calls.values()), {1})

`warm` asks ramabana for the backend of each task in the background, and sends nothing to the model. It gives the thread.
The jobs of `WARM` start too: the `guide` job, on the model of hints. `warm(['guide'])` starts only the guide.

In [ ]:
if has_rama:
    a, be = fake_agent(); seen = []
    a._be_or_none = lambda job='turn': seen.append(job) or be
    with using(a):
        Router(RouteAsk('claude/sonnet'), models=lambda: {}).warm().join()
    test_eq(seen, [*TASKS, 'guide'])
    test_eq(be.sent, [])
    seen.clear()
    with using(a): Router(RouteAsk('claude/sonnet'), models=lambda: dict(hint='claude/haiku')).warm(['guide']).join()
    test_eq((seen, a.routing.policy['guide']), (['guide'], 'claude/haiku'))

With ramabana, `route_ask` gives a `RouteAsk`. Without it, `route_ask` gives `rishi_ask` and a warning that says so one time.

In [ ]:
import sys, warnings
from unittest.mock import patch
if has_rama: assert isinstance(route_ask(), RouteAsk)
with patch.dict(sys.modules, {'ramabana': None}), warnings.catch_warnings(record=True) as w:
    warnings.simplefilter('always')
    ask = route_ask('claude/haiku')
test_eq(ask.model, 'claude/haiku')
assert not isinstance(ask, RouteAsk) and 'ramabana' in str(w[0].message)

### The reply cap of each task

A reply cap comes from the settings table (`tokens.<task>`), from 1 to `TOKENS_MAX`. A cap that you typed in the file as text, such as `"900"`, works too.

In [ ]:
from abhyasa.settings import update_settings, val
update_settings('tokens', hint=' 900 ')
test_eq(val('tokens.hint'), 900)
update_settings('tokens', hint=str(TOKENS_MAX + 1))
test_eq(val('tokens.hint'), TOKENS_MAX)   # clamped to the top
update_settings('tokens', hint=None)

`Router.cap` gives the saved cap of a task, else the default of the task. A bad value in the settings file gives the default, so a
call never fails because of it.

In [ ]:
caps = dict(hint=300)
r = Router(Fake(), models=lambda: {}, tokens=lambda: caps)
test_eq(r.cap('hint'), 300)
test_eq(r.cap('grade'), TOKENS['grade'])
caps['hint'] = 'lots'; test_eq(r.cap('hint'), TOKENS['hint'])
caps['hint'] = 99999; test_eq(r.cap('hint'), TOKENS_MAX)          # out of range: the most

A task after `extract` (`plan`, `pick`, `note`, `profile`, `exercise`) borrows the model, the reply cap and the job of an older task until it has a model of its own.

In [ ]:
class Via:
    model = 'base'
    def __call__(self, p, sp=''): return ''
    def via(self, job, name='', cap=None): return (job, name, cap)
ms = dict(extract='x/extract', hint='x/hint')
r = Router(Via(), models=lambda: ms, tokens=lambda: {})
test_eq((r.for_('plan'), r.for_('note'), r.for_('exercise')), (('extract', 'x/extract', 4096), ('hint', 'x/hint', 512), ('question', '', 4096)))
ms['plan'] = 'x/plan'
test_eq(r.for_('plan'), ('plan', 'x/plan', 4096))

The cap reaches `Agent.oneshot`, and a change has an effect on the next call.

In [ ]:
if has_rama:
    a, be = fake_agent()
    jobs, one = [], a.oneshot
    a.oneshot = lambda p, sp='', job='oneshot', max_tokens=None, fits=None: jobs.append((job, max_tokens)) or one(p, sp, job, max_tokens, fits)
    caps = dict(grade=777)
    with using(a):
        r = Router(RouteAsk('claude/sonnet'), models=lambda: {}, tokens=lambda: caps)
        r.for_('grade')('hi'); caps['grade'] = ''; r.for_('grade')('hi')
    test_eq(jobs, [('grade', 777), ('grade', TOKENS['grade'])])

The Test button of the Settings page asks on its own job, `test:<task>`, with the cap of the task. The job of the task stays where
the settings put it.

In [ ]:
if has_rama:
    a, be = fake_agent()
    jobs, one = [], a.oneshot
    a.oneshot = lambda p, sp='', job='oneshot', max_tokens=None, fits=None: jobs.append((job, max_tokens)) or one(p, sp, job, max_tokens, fits)
    with using(a):
        r = Router(RouteAsk('claude/sonnet'), models=lambda: dict(grade='claude/haiku'), tokens=lambda: {})
        r.for_('grade')('hi')
        r.for_('grade', 'claude/opus', job='test:grade')('hi')
    test_eq(jobs, [('grade', TOKENS['grade']), ('test:grade', TOKENS['grade'])])
    test_eq((a.routing.policy['grade'], a.routing.policy['test:grade']), ('claude/haiku', 'claude/opus'))

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()